# YOLO实例分割推理测试示例

本笔记本演示了如何使用推理测试脚本对单张图片或整个文件夹进行实例分割并可视化结果。

In [ ]:
# 导入必要的库
import sys
import os
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 添加项目路径
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "service"))

In [ ]:
# 从服务模块导入YOLOMaskService
from service.inference import YOLOMaskService

## 配置参数

In [ ]:
# 设置模型权重路径
WEIGHTS_PATH = "/home/industai/workspace/SubStation_AI_Poject/runs/segment/runs/train/exp/weights/best.pt"

# 设置数据配置文件路径（用于获取类别名称）
DATA_CONFIG_PATH = "/home/industai/workspace/SubStation_AI_Poject/runs/train/data.yaml"

# 设置设备（"0"表示GPU 0，"cpu"表示CPU）
DEVICE = "0"

# 设置置信度阈值
CONF_THRESHOLD = 0.25

# 设置输入图像尺寸
IMG_SIZE = 640

# 设置测试图片路径
TEST_IMAGE_PATH = "test_images/sample.jpg"  # 替换为你的测试图片路径
TEST_FOLDER_PATH = "test_images/"           # 替换为你的测试文件夹路径

## 初始化模型服务

In [ ]:
# 检查权重文件是否存在
if not Path(WEIGHTS_PATH).exists():
    print(f"警告: 权重文件不存在: {WEIGHTS_PATH}")
    # 如果找不到，列出可用的权重文件
    weights_files = list(Path(".").rglob("*/weights/*.pt"))
    if weights_files:
        print("找到的权重文件:")
        for wf in weights_files[:5]:  # 显示前5个
            print(f"  {wf}")
else:
    print(f"使用权重文件: {WEIGHTS_PATH}")

# 初始化模型服务
print("初始化模型服务...")
model_service = YOLOMaskService(
    weights_path=WEIGHTS_PATH,
    device=DEVICE,
    conf_threshold=CONF_THRESHOLD,
    img_size=IMG_SIZE
)
print("模型服务初始化完成")

## 从配置文件加载类别名称

In [ ]:
def load_class_names_from_yaml(data_yaml_path):
    """
    从yaml配置文件中加载类别名称
    """
    import yaml
    
    try:
        with open(data_yaml_path, 'r', encoding='utf-8') as f:
            data = yaml.safe_load(f)
        
        # 获取类别名称列表
        if 'names' in data:
            names_dict = data['names']
            if isinstance(names_dict, dict):
                # 如果是字典格式 {0: 'class1', 1: 'class2', ...}
                class_names = [''] * len(names_dict)
                for k, v in names_dict.items():
                    class_names[int(k)] = v
                return class_names
            elif isinstance(names_dict, list):
                # 如果是列表格式 ['class1', 'class2', ...]
                return names_dict
        
        return None
    except Exception as e:
        print(f"加载类别名称失败: {e}")
        return None

# 加载类别名称
class_names = None
if Path(DATA_CONFIG_PATH).exists():
    class_names = load_class_names_from_yaml(DATA_CONFIG_PATH)
    if class_names:
        print(f"加载了 {len(class_names)} 个类别名称:")
        for i, name in enumerate(class_names):
            print(f"  {i}: {name}")
else:
    print(f"数据配置文件不存在: {DATA_CONFIG_PATH}")

## 定义可视化函数

In [ ]:
def visualize_segmentation_result(image, detections, class_names=None):
    """
    可视化分割结果
    
    Args:
        image: 原始图像 (RGB格式)
        detections: 检测结果列表
        class_names: 类别名称列表
    """
    # 复制图像以便绘制
    vis_image = image.copy()
    
    for detection in detections:
        # 获取边界框和类别信息
        bbox = detection['bbox']
        confidence = detection['confidence']
        class_id = detection['class_id']
        contours = detection.get('contours', [])
        
        # 绘制边界框
        cv2.rectangle(vis_image, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), (0, 255, 0), 2)
        
        # 显示类别和置信度
        label = f"Class {class_id}: {confidence:.2f}"
        if class_names and class_id < len(class_names):
            label = f"{class_names[class_id]}: {confidence:.2f}"
        
        cv2.putText(vis_image, label, (int(bbox[0]), int(bbox[1]) - 10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        # 绘制分割轮廓
        for contour_info in contours:
            points = np.array(contour_info['points'], dtype=np.int32)
            if len(points) > 0:
                cv2.polylines(vis_image, [points], True, (255, 0, 0), 2)
                
                # 填充分割区域
                mask = np.zeros(vis_image.shape[:2], dtype=np.uint8)
                cv2.fillPoly(mask, [points], 255)
                
                # 创建彩色掩码层
                color_mask = np.zeros_like(vis_image)
                color_mask[:, :, 1] = 255  # 绿色通道
                masked_region = np.where(mask == 255)
                vis_image[masked_region[0], masked_region[1], :] = (
                    0.7 * vis_image[masked_region[0], masked_region[1], :].astype(np.float32) + 
                    0.3 * color_mask[masked_region[0], masked_region[1], :].astype(np.float32)
                ).astype(np.uint8)
    
    return vis_image

## 处理单张图片

In [ ]:
# 检查测试图片是否存在
if not Path(TEST_IMAGE_PATH).exists():
    print(f"测试图片不存在: {TEST_IMAGE_PATH}")
    # 尝试从runs目录中查找一些图片
    sample_images = list(Path(".").rglob("*.jpg")) + list(Path(".").rglob("*.png"))
    sample_images = [p for p in sample_images if 'runs' in str(p) and 'predict' in str(p)]
    if sample_images:
        TEST_IMAGE_PATH = str(sample_images[0])
        print(f"使用发现的图片: {TEST_IMAGE_PATH}")
    else:
        print("未找到任何图片文件，请提供有效的图片路径")
        
# 读取图像
image_bgr = cv2.imread(TEST_IMAGE_PATH)
if image_bgr is None:
    print(f"无法读取图片: {TEST_IMAGE_PATH}")
else:
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    
    # 执行预测
    print(f"处理图片: {TEST_IMAGE_PATH}")
    result = model_service.predict(image_rgb)
    
    # 获取检测结果
    all_detections = []
    for res in result['results']:
        all_detections.extend(res['detections'])
    
    # 可视化结果
    vis_image = visualize_segmentation_result(image_rgb, all_detections, class_names)
    
    # 显示结果
    plt.figure(figsize=(15, 8))
    plt.subplot(1, 2, 1)
    plt.title("原始图像")
    plt.imshow(image_rgb)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title(f"分割结果 - 检测到 {len(all_detections)} 个目标")
    plt.imshow(vis_image)
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"检测到 {len(all_detections)} 个目标")
    for i, det in enumerate(all_detections):
        class_id = det['class_id']
        conf = det['confidence']
        class_name = class_names[class_id] if class_names and class_id < len(class_names) else f"Class {class_id}"
        print(f"  目标 {i+1}: {class_name} (置信度: {conf:.3f})")

## 处理文件夹中的图片

In [ ]:
# 检查测试文件夹是否存在
if not Path(TEST_FOLDER_PATH).exists():
    print(f"测试文件夹不存在: {TEST_FOLDER_PATH}")
    # 尝试使用runs目录中的图片
    predict_dirs = [p for p in Path(".").rglob("runs/*") if 'predict' in str(p) and p.is_dir()]
    if predict_dirs:
        TEST_FOLDER_PATH = str(predict_dirs[0])
        print(f"使用发现的目录: {TEST_FOLDER_PATH}")
    else:
        print("未找到任何图片目录，请提供有效的图片文件夹路径")
else:
    # 获取文件夹中的所有图片
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']
    image_paths = []
    
    for ext in image_extensions:
        image_paths.extend(list(Path(TEST_FOLDER_PATH).rglob(f'*{ext}')))
        image_paths.extend(list(Path(TEST_FOLDER_PATH).rglob(f'*{ext.upper()}')))
    
    print(f"在文件夹 {TEST_FOLDER_PATH} 中找到 {len(image_paths)} 张图片")
    
    # 只处理前几张图片作为示例
    for idx, image_path in enumerate(image_paths[:3]):  # 只处理前3张
        print(f"\n处理第 {idx+1}/{min(3, len(image_paths))} 张图片: {image_path.name}")
        
        # 读取图像
        image_bgr = cv2.imread(str(image_path))
        if image_bgr is None:
            print(f"无法读取图片: {image_path}")
            continue
        
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        
        # 执行预测
        result = model_service.predict(image_rgb)
        
        # 获取检测结果
        all_detections = []
        for res in result['results']:
            all_detections.extend(res['detections'])
        
        # 可视化结果
        vis_image = visualize_segmentation_result(image_rgb, all_detections, class_names)
        
        # 显示结果
        plt.figure(figsize=(15, 8))
        plt.subplot(1, 2, 1)
        plt.title(f"原始图像 - {image_path.name}")
        plt.imshow(image_rgb)
        plt.axis('off')
        
        plt.subplot(1, 2, 2)
        plt.title(f"分割结果 - 检测到 {len(all_detections)} 个目标")
        plt.imshow(vis_image)
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        print(f"检测到 {len(all_detections)} 个目标")

## 总结

本笔记本展示了如何使用推理测试功能来处理单张图片或整个文件夹的图片，并可视化实例分割结果。您可以根据需要调整参数，如置信度阈值、输入图像尺寸等，以获得更好的分割效果。